# Vet record timeline

**What this notebook does:** reads the veterinary records of one pet, asks Gemini to list every
visit in them, and puts the visits from all the files into one timeline ordered by date.

**How it works, in order:**

1. The record files of one pet are found in BigQuery (query to be filled in) and downloaded to
   `data/records/<pet name or claim id>/`. PDF, JPG and PNG files are read.
2. Each file is sent to Gemini through Vertex AI (Google Cloud's service for running LLMs).
3. Gemini returns one entry per visit, in a fixed format (see "The output format" below).
4. The answer for each file is saved as a JSON file in `output/`.
5. The visits from all files are joined into one table, sorted by visit date.

**Before the first run:**

- Kernel: `.venv/bin/python` in this folder (created with `poetry install --no-root`).
- `gcloud auth application-default login` (signs this machine in to Google Cloud).
- Fill in the settings in the next cell.

In [ ]:
from pathlib import Path

# --- Settings: fill these in ---

# The Google Cloud project that pays for the Gemini calls. Not confirmed: check which project to use.
PROJECT_ID = "FILL-IN"

# The Google Cloud region the calls run in. Must be in the EU (gdpr-compliance.md RULE 6).
# Not confirmed that every Gemini version is offered in this region.
LOCATION = "europe-west1"

# The Gemini versions to compare. The extraction runs once per file for each name in this list.
MODELS = ["gemini-3.7-flash"]

# The folder with one pet's record files, and the folder the answers are saved to.
PET = "FILL-IN"
RECORDS_DIR = Path("data/records") / PET
OUTPUT_DIR = Path("output") / PET

In [ ]:
import json

import pandas as pd
from google import genai
from google.genai import types
from google.cloud import bigquery
from pydantic import BaseModel

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

## BigQuery


In [ ]:
# BigQuery client (your own Google login) and a query helper that dry-runs first
bq_client = bigquery.Client(project='hedvig-coruscant')


def run_query(sql, max_gb=10):
    dry = bq_client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = dry.total_bytes_processed / 1e9
    print(f'Dry run: this query scans {gb:.2f} GB')
    if gb > max_gb:
        raise ValueError(f'{gb:.2f} GB is above max_gb={max_gb}. Approve by raising max_gb.')
    return bq_client.query(sql).result().to_dataframe()

In [ ]:
# The query that lists the record files of one pet claim. Table names to be filled in.
SQL = """
-- FILL-IN
"""
# record_files_table = run_query(SQL)

## File access on AWS

The uploaded claim files are stored on AWS S3. `dbt_prod.stg_claims_service__claim_file` has one row
per uploaded file, with `bucket` and `key` giving the file's location in S3. The query below returns
the patient-history file of one claim, as the test case for reading a file from S3.

In [ ]:
CLAIM_ID = "e65784a6-5666-4e4b-9ff6-57c579d0abad"

SQL = f"""
SELECT claims_id, file_name, display_name, content_type, folder, bucket, key, uploaded_at
FROM `hedvig-coruscant.dbt_prod.stg_claims_service__claim_file`
WHERE claims_id = '{CLAIM_ID}'
  AND LOWER(file_name) LIKE '%patient_history%'
  AND NOT IFNULL(marked_as_deleted, FALSE)
"""
file_row = run_query(SQL)
file_row